# Modeling and tuning — Steps 11–12

**CSE437 Data Science | Group 15 | Owner: Sadat | Next: Step 13**

Step 11 compares the majority baseline and two model families. Step 12, below, tunes both learned families using the frozen development folds. The Step 11 cells and outputs are preserved as historical reference. The original dataset, target, problem/questions and evaluation split remain unchanged. Final test evaluation remains Step 13.

Execution: six preserved Step 11 Python-executed cells plus five newly Python-executed Step 12 cells. Separate fresh Jupyter-kernel execution and canonical notebook validation remain final gates.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
ROOT=Path.cwd().resolve()
if not (ROOT/"src").is_dir() and (ROOT.parent/"src").is_dir():
    ROOT=ROOT.parent
assert (ROOT/"data/splits/step6_split_plan.json").is_file()
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.model_comparison import PROTOCOL,run_model_comparison
from src.modeling import make_model_pipeline
print(json.dumps(PROTOCOL,indent=2))


{
  "candidates": [
    {
      "candidate": "majority",
      "family": "majority",
      "representation": "none"
    },
    {
      "candidate": "lr_full",
      "family": "logistic_regression",
      "representation": "full"
    },
    {
      "candidate": "lr_selected",
      "family": "logistic_regression",
      "representation": "selected"
    },
    {
      "candidate": "rf_full",
      "family": "random_forest",
      "representation": "full"
    },
    {
      "candidate": "rf_selected",
      "family": "random_forest",
      "representation": "selected"
    }
  ],
  "model_settings": {
    "logistic_regression": {
      "C": 1.0,
      "solver": "lbfgs",
      "max_iter": 2000,
      "tol": 0.0001,
      "class_weight": null,
      "random_state": 42
    },
    "random_forest": {
      "n_estimators": 100,
      "criterion": "gini",
      "max_depth": null,
      "min_samples_split": 2,
      "min_samples_leaf": 1,
      "max_features": "sqrt",
      "bootstrap": true,
    

## Models and validation

| Candidate | Purpose / fixed setup |
| --- | --- |
| Majority baseline | Predict the majority class observed in each training fold; ignores features |
| Logistic regression, full / selected | Linear regularized reference; C=1, lbfgs, max_iter=2000, tol=1e−4; scaled numeric inputs |
| Random forest, full / selected | Nonlinear tree interactions; 100 trees, unlimited depth, leaf size 1, sqrt feature sampling, bootstrap, no class weighting; unscaled numeric inputs |

Seeds are 42. Threshold is 0.5 (probability ≥0.5 predicts cancellation). No class resampling, threshold optimization or model-parameter search occurs. Three expanding development folds provide the same rows to each candidate. Every model gets a fresh pipeline; feature selection is fitted on training labels only. Mean cancellation F1 across the three folds is primary; accuracy, precision, recall and ROC-AUC are secondary.

The first run writes the comparison protocol before fitting and rejects unversioned protocol changes. Logistic metrics are checked against the published Step 10 reference. The random-forest setup is an untuned starting point; its depth and leaf size may permit overfitting, which the diagnostics below expose.


In [2]:
summary,comparison,fold_results=run_model_comparison(ROOT)
print("\nPreferred untuned candidate:",summary["preferred_candidate"])
print("Final test rows processed:",summary["test_rows_fitted_transformed_or_scored"])


Step 11: fitting majority, fold 1 (23,797 training rows)
Completed majority, fold 1: validation F1=0.000000; training F1=0.000000
Step 11: fitting lr_full, fold 1 (23,797 training rows)
Completed lr_full, fold 1: validation F1=0.643231; training F1=0.854044
Step 11: fitting lr_selected, fold 1 (23,797 training rows)
Completed lr_selected, fold 1: validation F1=0.693691; training F1=0.850240
Step 11: fitting rf_full, fold 1 (23,797 training rows)
Completed rf_full, fold 1: validation F1=0.657928; training F1=0.987615
Step 11: fitting rf_selected, fold 1 (23,797 training rows)
Completed rf_selected, fold 1: validation F1=0.667216; training F1=0.987439
Step 11: fitting majority, fold 2 (47,690 training rows)
Completed majority, fold 2: validation F1=0.000000; training F1=0.000000
Step 11: fitting lr_full, fold 2 (47,690 training rows)
Completed lr_full, fold 2: validation F1=0.702455; training F1=0.804477
Step 11: fitting lr_selected, fold 2 (47,690 training rows)
Completed lr_selected, f

## Development comparison

Scores describe the fixed development windows, not final test performance. The folds are reused for model/representation choice, so the winning mean can be optimistic. Across-fold SD is descriptive, not a confidence interval.

![Untuned model comparison](../figures/07_model_comparison.png)


In [3]:
print(comparison[["candidate","mean_f1","fold_sd_f1","mean_accuracy","mean_precision","mean_recall","mean_roc_auc"]].round(6).to_string(index=False))
print("\nF1 by forward validation period:")
print(fold_results.pivot(index="fold",columns="candidate",values="f1").round(6).to_string())


  candidate  mean_f1  fold_sd_f1  mean_accuracy  mean_precision  mean_recall  mean_roc_auc
   majority 0.000000    0.000000       0.638167        0.000000     0.000000      0.500000
    lr_full 0.693094    0.045904       0.751370        0.681025     0.753386      0.876098
lr_selected 0.713609    0.019669       0.809314        0.783364     0.659498      0.882905
    rf_full 0.626504    0.047179       0.793498        0.906662     0.481119      0.886473
rf_selected 0.657994    0.016997       0.803866        0.893717     0.521714      0.892267

F1 by forward validation period:
candidate   lr_full  lr_selected  majority   rf_full  rf_selected
fold                                                             
1          0.643231     0.693691       0.0  0.657928     0.667216
2          0.702455     0.714117       0.0  0.572253     0.638378
3          0.733595     0.733020       0.0  0.649330     0.668386


## Imbalance and training-versus-validation diagnostics

A majority baseline can have substantial accuracy while missing every cancellation. This explains why cancellation F1, recall and precision matter. Training scores are in-sample resubstitution diagnostics; gaps can reflect both overfitting and temporal shift, with repeated profiles adding another limitation. They are not independent generalization estimates.


In [4]:
print(comparison[["candidate","mean_training_f1","mean_f1","mean_f1_gap","mean_fit_seconds"]].round(6).to_string(index=False))
balance=fold_results.query("candidate=='majority'")[["fold","train_rows","training_canceled","validation_rows","validation_canceled","training_majority_class"]].copy()
balance["training_cancellation_percent"]=100*balance.training_canceled/balance.train_rows
balance["validation_cancellation_percent"]=100*balance.validation_canceled/balance.validation_rows
print("\nFold class composition (development only):")
print(balance.round(3).to_string(index=False))
print("\nAggregate confusion counts are saved per candidate/fold in fold_results.csv.")


  candidate  mean_training_f1  mean_f1  mean_f1_gap  mean_fit_seconds
   majority          0.000000 0.000000     0.000000          0.002263
    lr_full          0.817267 0.693094     0.124173          1.494870
lr_selected          0.814529 0.713609     0.100920          1.370380
    rf_full          0.990300 0.626504     0.363796         20.098629
rf_selected          0.990202 0.657994     0.332209         19.409045

Fold class composition (development only):
 fold  train_rows  training_canceled  validation_rows  validation_canceled  training_majority_class  training_cancellation_percent  validation_cancellation_percent
    1       23797               8561            23893                 8573                        0                         35.975                           35.881
    2       47690              17134            23776                 8867                        0                         35.928                           37.294
    3       71466              26001        

## Pipeline and reproduction checks

The saved evidence includes full estimator parameters, each fold's feature names, confusion counts, membership hashes and frozen input checksums. Prediction must not change the learned representation. Logistic results must match Step 10's full/selected reference. No fitted global model or row-level predictions are exported at this stage.


In [5]:
assert summary["model_fits"]==15 and summary["candidate_count"]==5
assert summary["logistic_results_match_step10"]
assert fold_results.representation_unchanged_after_prediction.all()
assert not fold_results.convergence_warning.any()
assert (fold_results[["tn","fp","fn","tp"]].sum(axis=1)==fold_results.validation_rows).all()
assert fold_results.groupby("fold").validation_membership_sha256.nunique().eq(1).all()
print("15 fits verified: identical validation membership within each fold and unchanged representation state.")
print("\nEncoded output widths:")
print(fold_results.pivot(index="fold",columns="candidate",values="encoded_columns").to_string())
print("\nLogistic metrics reproduce Step 10; frozen source/split checksums verified.")


15 fits verified: identical validation membership within each fold and unchanged representation state.

Encoded output widths:
candidate  lr_full  lr_selected  majority  rf_full  rf_selected
fold                                                           
1              332          247         0      332          247
2              421          314         0      421          314
3              490          366         0      490          366

Logistic metrics reproduce Step 10; frozen source/split checksums verified.


## Handoff — Step 12, Sadat

Use the leading untuned candidate as a starting point, not a final answer. Step 12 must report model search spaces, search method, candidate/fold counts and results for the two learned families. Keep the same development folds, primary metric and threshold policy. Representation preferences may differ by family; retain evidence for those choices.

Keep every learned preprocessing/selection step inside the pipeline passed to the search. Do not use a globally preprocessed matrix, validation labels to fit selectors, or held-out scores to choose settings. Freeze all choices before Step 13's final refit/test evaluation.


In [6]:
next_pipeline=make_model_pipeline(summary["preferred_family"],summary["preferred_representation"])
assert not hasattr(next_pipeline.named_steps.get("representation"),"preprocessor_")
assert summary["test_rows_fitted_transformed_or_scored"]==0
assert summary["model_hyperparameter_tuning_completed"] is False
print("Step 11 complete. Current untuned leader:",summary["preferred_candidate"])
print("Step 12 — Sadat: train-fold model hyperparameter search; test still reserved for Step 13.")


Step 11 complete. Current untuned leader: lr_selected
Step 12 — Sadat: train-fold model hyperparameter search; test still reserved for Step 13.


# Step 12 — Hyperparameter tuning

**Owner: Sadat | Next: Step 13 — final test evaluation and error analysis**

The preceding Step 11 experiment is preserved as the untuned reference. This section conducts an exhaustive grid search for both learned families. The three development folds, selected-feature rule (75%), and cancellation threshold (0.5) stay fixed. No final test data is used, and no best estimator is refitted on all development data in this step.

| Family | Search space | Settings × folds |
| --- | --- | ---: |
| Logistic regression | C ∈ {0.01, 0.1, 1, 10}; class_weight ∈ {None, balanced} | 8 × 3 = 24 |
| Random forest | max_depth ∈ {8, 16, None}; min_samples_leaf ∈ {1, 10}; class_weight ∈ {None, balanced} | 12 × 3 = 36 |

Logistic solver/max_iter remain lbfgs/2000. Forest size remains 100 trees, max_features=sqrt, bootstrap=True; seed 42 remains fixed. This bounded grid examines regularization and class imbalance without claiming global optimality. Class weights are learned from each training fold, not from validation or test labels. Both grids include the Step 11 selected-feature control.


In [7]:
from pathlib import Path
import sys, json
import numpy as np
ROOT=Path.cwd().resolve()
if not (ROOT/"src").is_dir() and (ROOT.parent/"src").is_dir(): ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.tuning import PROTOCOL as TUNING_PROTOCOL, run_tuning, rank_candidates, build_frozen_pipeline
print(json.dumps(TUNING_PROTOCOL,indent=2))


{
  "step": 12,
  "method": "exhaustive GridSearchCV",
  "parameter_grids": {
    "logistic_regression": {
      "model__C": [
        0.01,
        0.1,
        1.0,
        10.0
      ],
      "model__class_weight": [
        null,
        "balanced"
      ]
    },
    "random_forest": {
      "model__max_depth": [
        8,
        16,
        null
      ],
      "model__min_samples_leaf": [
        1,
        10
      ],
      "model__class_weight": [
        null,
        "balanced"
      ]
    }
  },
  "representations": {
    "logistic_regression": "selected",
    "random_forest": "selected"
  },
  "selection_percentile": 75,
  "pca_retained": false,
  "fixed_settings_source": "src/modeling.py MODEL_SETTINGS; only listed model parameters vary",
  "candidate_counts": {
    "logistic_regression": 8,
    "random_forest": 12
  },
  "folds": 3,
  "expected_fits": 60,
  "cv": "immutable Step 6 expanding forward folds; development-relative indices",
  "seed": 42,
  "search_n_jobs": 1,

## Search and scoring

`GridSearchCV` clones a complete raw-input pipeline for every setting/fold. Imputation, encoding, scaling and supervised selection are fitted inside that training fold. Search runs sequentially with two workers per forest. A custom multimetric scorer uses cancellation probability ≥0.5, including ties, rather than relying on estimator-specific `predict` behavior.

Mean cancellation F1 across the three folds selects the winner. Exact ties follow the frozen family/ParameterGrid order. Accuracy, precision, recall, ROC-AUC and confusion counts are also recorded. `refit=False` preserves the Step 13 boundary; failed fits or convergence warnings stop the run instead of silently accepting incomplete evidence. The protocol is persisted before new scores are computed.


In [8]:
tuning_summary,tuning_scores,tuning_folds,tuning_comparison,selection=run_tuning(ROOT)
print("Completed candidate settings:",tuning_summary["candidates"])
print("Completed fold fits:",tuning_summary["model_fits"])
print("Global development refit:",tuning_summary["full_development_model_fitted"])



Step 12: logistic_regression, 8 settings × 3 frozen folds
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END ............model__C=0.01, model__class_weight=None; total time=   0.7s
[CV] END ............model__C=0.01, model__class_weight=None; total time=   1.3s
[CV] END ............model__C=0.01, model__class_weight=None; total time=   1.8s
[CV] END ........model__C=0.01, model__class_weight=balanced; total time=   0.7s
[CV] END ........model__C=0.01, model__class_weight=balanced; total time=   1.3s
[CV] END ........model__C=0.01, model__class_weight=balanced; total time=   1.8s
[CV] END .............model__C=0.1, model__class_weight=None; total time=   0.7s
[CV] END .............model__C=0.1, model__class_weight=None; total time=   1.3s
[CV] END .............model__C=0.1, model__class_weight=None; total time=   2.1s
[CV] END .........model__C=0.1, model__class_weight=balanced; total time=   0.7s
[CV] END .........model__C=0.1, model__class_weight=balanced; total tim

## Actual search results

The table contains every setting; full search outputs and exact parameters are saved in `data/results/step12/`. Raw sklearn CV tables report population SD; the readable candidate table uses sample SD across the three folds. Neither is a confidence interval. Training scores are resubstitution diagnostics.

![Untuned versus tuned development F1](../figures/08_hyperparameter_tuning.png)


In [9]:
columns=["candidate","family","parameters_json","mean_f1","fold_sd_f1","mean_accuracy","mean_precision","mean_recall","mean_roc_auc","mean_training_f1","mean_f1_gap"]
print(rank_candidates(tuning_scores)[columns].round(6).to_string(index=False))
print("\nSame-fold comparison against Step 11:")
print(tuning_comparison.round(6).to_string(index=False))


candidate              family                                                                              parameters_json  mean_f1  fold_sd_f1  mean_accuracy  mean_precision  mean_recall  mean_roc_auc  mean_training_f1  mean_f1_gap
    lr_06 logistic_regression                                         {"model__C": 1.0, "model__class_weight": "balanced"} 0.732102    0.016451       0.800244        0.715322     0.756772      0.882666          0.827779     0.095677
    lr_04 logistic_regression                                         {"model__C": 0.1, "model__class_weight": "balanced"} 0.731697    0.020592       0.800329        0.715143     0.755522      0.884911          0.824861     0.093164
    lr_08 logistic_regression                                        {"model__C": 10.0, "model__class_weight": "balanced"} 0.730446    0.020174       0.798557        0.711247     0.757295      0.880002          0.828608     0.098162
    lr_02 logistic_regression                                       

## Interpretation and verification

Compare the best grid setting within each family with its untuned control, including recall/precision and the training–validation gap. Balanced weighting can improve cancellation recall while increasing false positives; the F1 objective determines this project's choice. Reusing the folds for representation and hyperparameter selection can make winning scores optimistic. Do not treat improvement as a confidence-tested effect or a final-test gain. Keep the documented bounded search even if a winning value lies at an edge; do not chase a particular score.


**Numerical audit:** the first complete grid run stopped on a strict secondary forest ROC-AUC comparison (maximum difference 1.14e−8); threshold-based metrics matched exactly. Such a difference is consistent with floating-point accumulation near tied ranks. Forest AUC parity now uses a documented 1e−7 tolerance; all other control metrics use 1e−12. Scoring, grids, threshold and model selection are unchanged. The displayed results come from the complete rerun. `control_parity.csv` preserves the measured differences; all 47 tests pass.


In [10]:
assert tuning_summary["model_fits"]==60 and tuning_summary["candidates"]==20
assert tuning_summary["untuned_controls_match_step11"]
assert tuning_summary["failed_fits"]==0 and tuning_summary["convergence_warnings"]==0
assert (tuning_folds[["tn","fp","fn","tp"]].sum(axis=1)==tuning_folds.validation_rows).all()
assert tuning_folds.groupby("fold").validation_membership_sha256.nunique().eq(1).all()
assert tuning_summary["test_rows_fitted_transformed_or_scored"]==0
print("60 fits verified; frozen inputs, shared validation membership and untuned-control parity verified.")
for family,chosen in selection["best_by_family"].items():
    print("\nBest setting for",family,":",chosen["search_parameters"])
    print(tuning_folds.loc[tuning_folds.candidate.eq(chosen["candidate"]),["fold","f1","precision","recall","roc_auc","training_f1"]].round(6).to_string(index=False))


60 fits verified; frozen inputs, shared validation membership and untuned-control parity verified.

Best setting for logistic_regression : {'model__C': 1.0, 'model__class_weight': 'balanced'}
 fold       f1  precision   recall  roc_auc  training_f1
    1 0.715132   0.750064 0.683308 0.874543     0.851066
    2 0.747979   0.739528 0.756626 0.887599     0.824261
    3 0.733194   0.656372 0.830382 0.885856     0.808009

Best setting for random_forest : {'model__class_weight': 'balanced', 'model__max_depth': None, 'model__min_samples_leaf': 10}
 fold       f1  precision   recall  roc_auc  training_f1
    1 0.669006   0.929461 0.522571 0.885465     0.889730
    2 0.648446   0.871195 0.516409 0.893679     0.873085
    3 0.690429   0.827751 0.592186 0.889960     0.854118


## Frozen handoff — Step 13, Sadat

`final_selection.json` records the development-selected family, exact estimator parameters, selected-feature rule, threshold and data lineage. The function below constructs an **unfitted** pipeline and rejects parameters outside the frozen search space or changed estimator defaults. Step 13 will refit on development only and then evaluate the untouched test once; test scores must not select settings.

**Execution provenance:** the six Step 11 cell outputs are preserved. These five new Step 12 cells were executed sequentially in a fresh Python process with actual stdout captured. No full 11-cell Jupyter-kernel run or canonical nbformat validation is claimed; both remain final submission gates. ChatGPT/Codex assisted; Sadat should review and record actual contributions.


In [11]:
frozen_pipeline=build_frozen_pipeline(selection)
assert not hasattr(frozen_pipeline.named_steps["representation"],"preprocessor_")
print("Selected family:",selection["family"])
print("Selected model parameters:",selection["search_parameters"])
print("Development mean cancellation F1:",round(selection["mean_development_f1"],6))
print("Threshold:",selection["threshold"])
print("Step 12 complete. Next: Step 13 — Sadat. Final test performance remains unknown.")


Selected family: logistic_regression
Selected model parameters: {'model__C': 1.0, 'model__class_weight': 'balanced'}
Development mean cancellation F1: 0.732102
Threshold: 0.5
Step 12 complete. Next: Step 13 — Sadat. Final test performance remains unknown.
